## **BOILER PLATE**

In [2]:
import os
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv(dotenv_path="../.env",override=True)

api_key=os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is missing from .env")

llm = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash-0731",
    api_key=api_key,
    temperature=1.2,
    max_tokens=500,
)

## **CHAIN WITH PARALLEL PROCESSINGS**

In [3]:
#TASK-1 prompt

from langchain_core.prompts import ChatPromptTemplate

prompt_template_1 = ChatPromptTemplate([
    ("system","You are a movie summarizer"),
    ("human","Please summarize this movie in brief:{input}")
])

In [4]:
#TASK-2 llm

llm_1 = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash-0731",
    api_key=api_key,
    temperature=0,
    max_tokens=2000,
)

In [5]:
#TASK-3 String Parser

from langchain_core.output_parsers import StrOutputParser

str_parser_1 = StrOutputParser()

In [7]:
#TASK-4  Custom Runnable
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)-> dict:
    return {"text": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)


### **PARALLEL CHAIN - 1**

In [12]:
from langchain_core.runnables import RunnableSequence,RunnableLambda

#TASK-1 prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a LinkedIn Social Media Manager"),
    ("human","Create a post for the following text: {text}")
])

#TASK-2 llm

llm_2 = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash-0731",
    api_key=api_key,
    temperature=1.0,
    max_tokens=20000,
)

#TASK-3 parser

str_parser_2 = StrOutputParser()

#TASK-4 chain

chain_linkedin = RunnableSequence(linkedin_prompt,llm_2,str_parser_2)

### **PARALLEL CHAIN 2**

In [13]:
def instagram_chain(text:dict):
    
    text = text["text"]

    #TASK-1 prompt

    instagram_prompt = ChatPromptTemplate.from_messages([
        ("system","You are a Instagram Social Media Manager"),
        ("human","Create a post for the following text: {text}")
    ])

    #TASK-2 llm

    llm_3 = ChatOpenRouter(
        model="deepseek/deepseek-v4-flash-0731",
        api_key=api_key,
        temperature=1.2,
        max_tokens=20000,
    )

    #TASK-3 parser

    str_parser_3 = StrOutputParser()

    #TASK-4 chain

    chain_instagram = RunnableSequence(instagram_prompt,llm_3,str_parser_3)

    result = chain_instagram.invoke(text)

    return result

instagram_chain_runnable = RunnableLambda(instagram_chain)

## **FINAL CHAIN**

In [ ]:
from langchain_core.runnables import RunnableParallel

final_chain = RunnableSequence(
    prompt_template_1,
    llm_1,
    str_parser_1,
    dictionary_maker_runnable,
    RunnableParallel(
        branches={
            "linkedin": chain_linkedin,
            "instagram": instagram_chain_runnable
        }
    )
)

{'branches': {'linkedin': 'Here’s a LinkedIn-ready post that transforms the *Endgame* summary into a leadership and teamwork lesson:  \n\n---\n\n**Five years after a catastrophic failure, the remaining Avengers didn’t quit—they pivoted.**  \n\nIn *Avengers: Endgame*, we watch a team that was shattered by the Snap discover an unconventional path forward: Scott Lang’s Quantum Realm time-travel theory. This is a masterclass in resilience, collaboration, and succession—and the corporate world has plenty to learn from it.  \n\n**Key takeaways:**  \n\n🔹 **No setback is final until you say it is.** The Avengers spent five years in despair, but the moment a new solution emerged, they re-strategized.  \n\n🔹 **Innovation often comes from unexpected places.** It wasn’t a general or a scientist—it was Ant-Man who cracked the code. Stay open to ideas from every level.  \n\n🔹 **Divide and conquer, but unite for the mission.** Each hero traveled to retrieve a different Infinity Stone, leaning on thei

## **CHAIN AS A RUNNABLE**

In [17]:
#TASK 1 - beautify function

def beautify(final_response:dict)-> dict:

    final_response = final_response["branches"]

    linkedin_response = final_response["linkedin"]
    instagram_response = final_response["instagram"]

    return {
        "linkedin":linkedin_response,
        "instagram":instagram_response
    } 

beautify_runnable = RunnableLambda(beautify)

#TASK 2 - beautified chain

beautified_chain = RunnableSequence(final_chain,beautify_runnable)

beautified_chain.invoke("The Amazing Spiderman")

{'linkedin': " 🎬 *The Amazing Spider-Man* isn’t just a superhero origin story—it’s a masterclass in embracing your identity and rising to responsibility when life throws you the unexpected.\n\nPeter Parker starts as a high school outcast, searching for answers about his missing father. That search leads him to Oscorp, a genetically altered spider, and extraordinary new abilities. But with great power comes greater questions: Who is he without the mask? And what does it truly mean to protect those he loves?\n\nWhen his father’s former partner, Dr. Curt Connors, transforms into the Lizard, Peter is forced to move beyond revenge and self-doubt. He must decide between playing it safe or owning his role as a hero—not just to save New York, but to uncover the truth about his past.\n\n💡 The takeaway? Sometimes our greatest challenges aren’t the villains we fight, but the uncertainty we carry. Growth happens when we stop running from who we are and start showing up for what matters.\n\nWhether